# 01 — Crossref data collection

In [1]:
import requests
import json
import re
import pandas as pd

from typing import Any, Dict, List
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# %%
# Configuration

CROSSREF_WORKS_URL = "https://api.crossref.org/works"

USER_AGENT = "SriLankaCollector/1.0 (mailto:your_email@example.com)"

SRI_LANKA_QUERIES = [ "ceylon", "lanka"]


In [2]:
def create_session(user_agent: str = USER_AGENT) -> requests.Session:
    """
    Create reusable HTTP session
    with retry support.
    """

    retry_strategy = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
    )
    session = requests.Session()

    adapter = HTTPAdapter(max_retries=retry_strategy)

    session.mount("https://", adapter)

    session.headers.update({"User-Agent": user_agent})

    return session


In [3]:
def normalize_text(text: str) -> str:

    return re.sub(r"\s+", " ", text).strip().casefold()


def has_sri_lankan_affiliation(
    work: Dict[str,Any]
) -> bool:
    """
    Check whether any author affiliation
    explicitly contains Sri Lanka related terms.
    """

    keywords = [
        "ceylon",
        "lanka"
    ]


    for author in work.get(
        "author",
        []
    ):
        for affiliation in author.get(
                "affiliation",
                []
            ):

                name = normalize_text(
                    affiliation.get(
                        "name",
                        ""
                    )
                )


                if any(
                    keyword in name
                    for keyword in keywords
                ):
                    return True


    return False


In [4]:
def fetch_works(
    session: requests.Session, query: str, max_records: int = 2000, rows: int = 1000
) -> List[Dict[str, Any]]:

    cursor = "*"

    records = []

    while len(records) < max_records:
        page_size = min(rows, max_records - len(records))
        response = session.get(
            CROSSREF_WORKS_URL,
            params={
                "query.affiliation": query,
                "rows": page_size,
                "cursor": cursor,
                "sort": "created",
                "order": "desc"
            },
            timeout=60
        )


        response.raise_for_status()


        message = response.json()["message"]

        items = message.get("items", [])

        if not items:
            break

        records.extend(items)

        print(f"{query}: collected {len(records)}")

        cursor = message.get("next-cursor")

        if not cursor or len(items) < page_size:
            break

    return records

def collect_sri_lanka_records(
    queries: List[str],
    max_records_per_query: int = 2000
) -> List[Dict[str,Any]]:


    session = create_session()


    unique_records = {}


    for query in queries:


        candidates = fetch_works(
            session,
            query,
            max_records_per_query
        )


        print(f"Validating {query} results...")

        for work in candidates:
            doi = work.get("DOI")

            if doi and has_sri_lankan_affiliation(work):
                unique_records[doi.casefold()] = work

    records = list(unique_records.values())

    print("Final unique Sri Lankan records:", len(records))

    return records

 



In [5]:
records = collect_sri_lanka_records(SRI_LANKA_QUERIES, max_records_per_query=2000)


KeyboardInterrupt: 

In [ ]:
with open("sri_lanka_crossref_raw.json", "w", encoding="utf-8") as file:
    json.dump(records, file, ensure_ascii=False, indent=2)


print("Raw JSON saved")


Raw JSON saved


In [ ]:
with open("sri_lanka_crossref_raw.json","r",encoding="utf-8") as file:
    data = json.load(file)

df=pd.json_normalize(data)
df.to_csv("crossref_flatten_s1.csv",index=False,encoding="utf-8")

print("csv saved")
print(df.shape)

csv saved
(2470, 83)


In [ ]:
relation_columns=[col for col in df.columns if col.startswith("relation")]
relation_columns

relation_columns = [col for col in df.columns if col.startswith("relation")]

reduced_df = df.drop(columns=relation_columns)

print(f"Dropped {len(relation_columns)} relation columns")
print(reduced_df.shape)

Dropped 7 relation columns
(2470, 76)


In [ ]:
reduced_df = reduced_df.drop(
    columns=[
        "published-online.date-parts",
        "published-print.date-parts",
        "created.date-parts",
        "deposited.date-parts",
        "indexed.date-parts",
        "source",
        "clinical-trial-number",
        "update-to",
        "updated-by",
        "special_numbering",
        "resource.primary.URL",
        "resource.secondary",
        "aliases",
        "archive",
        "journal-issue.published-online.date-parts",
        "assertion",
        "published-other.date-parts",
        "posted.date-parts",
        "accepted.date-parts",
        "archieve",
        "short-container-title",
        "journal-issue.published-print.date-parts",
        "update-policy",
        "links",
        "indexed.version",
        "indexed.timestamp",
        "indexed.date-time",
        "created.timestamp",
        "created.date-time",
        "deposited.timestamp",
        "deposited.date-time",
        "score",
        "prefix",
        "references-count"
    ],
    errors="ignore",
)

reduced_df.shape


(2470, 44)

In [ ]:
cols=reduced_df.isna().sum()
cols

reference-count                            0
publisher                                  1
issue                                   1612
abstract                                1177
DOI                                        0
type                                       0
is-referenced-by-count                     0
title                                      0
volume                                  1528
author                                     0
member                                     1
container-title                           57
URL                                        0
ISSN                                    1369
issn-type                               1369
content-domain.domain                      0
content-domain.crossmark-restriction       0
issued.date-parts                          0
journal-issue.issue                     1612
published.date-parts                       0
license                                  911
page                                     360
reference 

In [ ]:
reduced_df.columns

Index(['reference-count', 'publisher', 'issue', 'abstract', 'DOI', 'type',
       'is-referenced-by-count', 'title', 'volume', 'author', 'member',
       'container-title', 'URL', 'ISSN', 'issn-type', 'content-domain.domain',
       'content-domain.crossmark-restriction', 'issued.date-parts',
       'journal-issue.issue', 'published.date-parts', 'license', 'page',
       'reference', 'link', 'event.name', 'event.location',
       'event.start.date-parts', 'event.end.date-parts', 'language', 'editor',
       'alternative-id', 'funder', 'article-number', 'isbn-type', 'ISBN',
       'publisher-location', 'event.acronym', 'institution', 'group-title',
       'subtype', 'event.sponsor', 'original-title', 'subtitle',
       'edition-number'],
      dtype='str')

In [ ]:
reduced_df["type"].unique()

<StringArray>
[    'journal-article', 'proceedings-article',        'book-chapter',
      'posted-content',              'report',           'monograph',
                'book',             'dataset']
Length: 8, dtype: str

In [ ]:
keep_types=["journal-article","proceedings-article","posted-content"]
reduced_df=reduced_df[reduced_df["type"].isin(keep_types)]

In [ ]:
reduced_df["type"].value_counts()

type
proceedings-article    1296
journal-article        1078
posted-content           54
Name: count, dtype: int64

In [ ]:
cols=reduced_df.columns
print(len(cols))
cols

44


Index(['reference-count', 'publisher', 'issue', 'abstract', 'DOI', 'type',
       'is-referenced-by-count', 'title', 'volume', 'author', 'member',
       'container-title', 'URL', 'ISSN', 'issn-type', 'content-domain.domain',
       'content-domain.crossmark-restriction', 'issued.date-parts',
       'journal-issue.issue', 'published.date-parts', 'license', 'page',
       'reference', 'link', 'event.name', 'event.location',
       'event.start.date-parts', 'event.end.date-parts', 'language', 'editor',
       'alternative-id', 'funder', 'article-number', 'isbn-type', 'ISBN',
       'publisher-location', 'event.acronym', 'institution', 'group-title',
       'subtype', 'event.sponsor', 'original-title', 'subtitle',
       'edition-number'],
      dtype='str')

In [ ]:
comparison = reduced_df[["issue", "journal-issue.issue"]].dropna(how="all")


(comparison["issue"]==comparison["journal-issue.issue"]).all()


np.True_

In [ ]:
reduced_df=reduced_df.drop(columns=["journal-issue.issue","alternative-id","issn-type","institution"],errors="ignore")

print(reduced_df.shape)

reduced_df.isna().sum().sort_values(ascending=False)



(2428, 33)


event.sponsor             2422
editor                    2419
original-title            2416
subtitle                  2401
publisher-location        2399
group-title               2380
subtype                   2374
article-number            2188
funder                    2185
event.acronym             2169
language                  1633
event.end.date-parts      1585
event.start.date-parts    1585
event.location            1571
issue                     1570
volume                    1486
ISSN                      1340
abstract                  1166
event.name                1132
reference                  886
license                    870
page                       354
container-title             54
publisher                    1
reference-count              0
issued.date-parts            0
published.date-parts         0
author                       0
URL                          0
DOI                          0
type                         0
is-referenced-by-count       0
title   

In [ ]:
print(reduced_df["group-title"].value_counts())
reduced_df[["group-title", "title"]].dropna().head(20)

group-title
In Review                                                            33
Ecology                                                               2
Bioinformatics                                                        2
Infectious Diseases (except HIV/AIDS)                                 2
Endocrinology (including Diabetes Mellitus and Metabolic Disease)     1
Primary Care Research                                                 1
searchRxiv                                                            1
Nephrology                                                            1
Addiction Medicine                                                    1
Genetics                                                              1
Orthopedics                                                           1
Pharmacology and Toxicology                                           1
Health Systems and Quality Improvement                                1
Name: count, dtype: int64


,group-title,title
59,In Review,[The assessment of the dietary patterns of pat...
113,In Review,[Effects of Nutritional Supplementation on Apo...
148,In Review,[An Effective Stability Solution for Small Pow...
187,In Review,[Air Quality Impact from Petroleum Refinery at...
234,In Review,[Eco Friendly Zinc Oxide Wurtzite Nanostructur...
526,In Review,[An Equivalence-Checked RV32IM Pipeline with I...
543,In Review,"[Population Ageing, Non-Communicable Disease B..."
555,Ecology,"[Lack of co-ordination of stomatal, hydraulic ..."
615,In Review,[Baseline LDL-C Influences Achievement of Cont...
637,In Review,[From Beliefs to Behaviour: A Scoping Review o...


In [ ]:
reduced_df[["title","subtitle"]].dropna().head(20)


,title,subtitle
108,[Speed-up Line Detection Approach for Large-si...,[]
139,[The Eosinophilic Respiratory Syndrome],[A Review of 100 Cases]
276,[ANEURYSM OF THE POPLITEAL ARTERY FROM PERFORA...,[Report of a Case]
283,[Police Patrol Systems],[Hours of Duty in a Crown Colony]
307,[Cyberattacks on Critical Infrastructure and P...,[]
331,[A comprehensive planning framework for electr...,[EV CHARGING INFRASTRUCTURE PLANNING]
337,[Wolff-Parkinson-White Syndrome],[Conversion of Type A to Type B Electrocardiog...
359,[Ceylon],[(Read at a Meeting of the Society in Edinburg...
391,[Book reviews : India's Simmering Revolution: ...,"[By SUMANTA BANERJEE (London, Zed Press, 1984)..."
398,[Ayurveda Medicine],[The Strange and Fascinating Tale of the Art a...


In [ ]:

print(reduced_df["ISSN"].isna().sum())
print(reduced_df["ISSN"].value_counts())

1340
ISSN
[2454-6186, 2454-6186]    52
[0007-1323, 1365-2168]    33
[2321-2705, 2321-2705]    27
[1029-8479]               21
[0374-5481]               18
                          ..
[2673-7248]                1
[1593-098X]                1
[1727-9232, 1810-3057]     1
[1081-5589, 1708-8267]     1
[2376-9637, 0015-7473]     1
Name: count, Length: 586, dtype: int64


In [ ]:
finalized_cols=reduced_df.columns
print(len(finalized_cols))
finalized_cols

33


Index(['reference-count', 'publisher', 'issue', 'abstract', 'DOI', 'type',
       'is-referenced-by-count', 'title', 'volume', 'author',
       'container-title', 'URL', 'ISSN', 'issued.date-parts',
       'published.date-parts', 'license', 'page', 'reference', 'event.name',
       'event.location', 'event.start.date-parts', 'event.end.date-parts',
       'language', 'editor', 'funder', 'article-number', 'publisher-location',
       'event.acronym', 'group-title', 'subtype', 'event.sponsor',
       'original-title', 'subtitle'],
      dtype='str')

In [ ]:
reduced_df.iloc[102]["container-title"]

['2017 Seventeenth International Conference on Advances in ICT for Emerging Regions (ICTer)']

In [ ]:
reduced_df.iloc[102]["event.name"]

'2017 Seventeenth International Conference on Advances in ICT for Emerging Regions (ICTer)'

In [ ]:
comparison["container-title"] = comparison["container-title"].apply(
    lambda x: x[0] if isinstance(x, list) else x
)

comparison["event.name_clean"] = (
    comparison["event.name"].astype(str).str.strip().str.casefold()
)

comparison["container-title_clean"] = (
    comparison["container-title"].astype(str).str.strip().str.casefold()
)


diff=(comparison["event.name_clean"]
    != comparison["container-title_clean"])

diff

1       False
2       False
3       False
4       False
5       False
        ...  
2451    False
2452     True
2453     True
2454     True
2463    False
Length: 1296, dtype: bool

In [ ]:
different = comparison[
    comparison["event.name_clean"] != comparison["container-title_clean"]
]

different


,container-title,event.name,event.name_clean,container-title_clean
6,2026 International Conference on Intelligent a...,2026 International Conference on Intelligent a...,2026 international conference on intelligent a...,2026 international conference on intelligent a...
38,2023 17th International Conference on Signal-I...,2023 17th International Conference on Signal-I...,2023 17th international conference on signal-i...,2023 17th international conference on signal-i...
39,Proceedings of the 2023 7th International Conf...,ICCBB 2023: 2023 7th International Conference ...,iccbb 2023: 2023 7th international conference ...,proceedings of the 2023 7th international conf...
50,Short Oral Presentation,The 15th International Congress on Systemic Lu...,the 15th international congress on systemic lu...,short oral presentation
51,Oral Presentation,The 15th International Congress on Systemic Lu...,the 15th international congress on systemic lu...,oral presentation
...,...,...,...,...
2346,Proceedings of the 19th International Joint Co...,Special Session on The emerging Dual-imaging s...,special session on the emerging dual-imaging s...,proceedings of the 19th international joint co...
2385,Posters,"15th European Lupus Meeting, Lisbon, Portugal,...","15th european lupus meeting, lisbon, portugal,...",posters
2452,2026 International Conference on Intelligent a...,2026 International Conference on Intelligent a...,2026 international conference on intelligent a...,2026 international conference on intelligent a...
2453,2026 International Conference on Intelligent a...,2026 International Conference on Intelligent a...,2026 international conference on intelligent a...,2026 international conference on intelligent a...


In [ ]:
reduced_df.iloc[50]["event.name"]

'The 15th International Congress on Systemic Lupus Erythematosus and The 43rd KCR Annual Scientific Meeting & 17th International Symposium (LUPUS & KCR 2023)'

In [ ]:
reduced_df.iloc[50]["container-title"]

['Oral Presentation']

In [ ]:
finalized_cols

Index(['reference-count', 'publisher', 'issue', 'abstract', 'DOI', 'type',
       'is-referenced-by-count', 'title', 'volume', 'author',
       'container-title', 'URL', 'ISSN', 'issued.date-parts',
       'published.date-parts', 'license', 'page', 'reference', 'event.name',
       'event.location', 'event.start.date-parts', 'event.end.date-parts',
       'language', 'editor', 'funder', 'article-number', 'publisher-location',
       'event.acronym', 'group-title', 'subtype', 'event.sponsor',
       'original-title', 'subtitle'],
      dtype='str')

In [ ]:
reduced_df["issued.date-parts"].isna().sum()

np.int64(0)

0       [{'given': 'Navaratnarajah', 'family': 'Sathip...
1       [{'given': 'R.M.D.E.', 'family': 'Rajapaksha',...
2       [{'given': 'Gimashi', 'family': 'Ishara', 'seq...
3       [{'given': 'Sankavi', 'family': 'Mohan', 'sequ...
4       [{'given': 'Jeyagopal', 'family': 'Janarththan...
                              ...                        
2464    [{'given': 'V Pujitha', 'family': 'Wickramasin...
2465    [{'name': 'CCAA Priority Setting Partnership S...
2466    [{'given': 'Malmi', 'family': 'Wickramasinghe'...
2467    [{'given': 'Pradeep Udaya', 'family': 'Rathnay...
2468    [{'name': 'The CMS collaboration', 'sequence':...
Name: author, Length: 2428, dtype: object

NameError: name 'df_cr' is not defined